# CellposeSAM segmentation across an AVITI24 cytoprofiling run

Run [CellposeSAM (CPSAM)](https://github.com/mouseland/cellpose) across every tile in a cytoprofiling run and write the segmentation masks Cells2Stats expects. CPSAM is a general-purpose model that integrates the [Segment Anything Model](https://segment-anything.com/) architecture; use it when your cell type is not represented in the Element Biosciences model library or when the General Element Biosciences model produces poor results even after diameter tuning.

CPSAM supports both **2-channel** (Cell-Membrane + Nucleus) and **3-channel** (Cell-Membrane + Nucleus + Actin) input. Use 3-channel mode for Teton and Teton Atlas runs where an actin channel is available; use 2-channel mode for Cell Paint only runs or any run without actin images.

This notebook is the companion to the [Custom segmentation tutorial](https://docs.elembio.io/docs/tutorials/cytoprofiling/custom-segmentation/). Read the tutorial first for the full context: run-type identification, when to use CPSAM, and post-segmentation Cells2Stats re-run.

## Prerequisites

Confirm the following before running this notebook:

- You created the separate `cpsam` Python environment with Cellpose 4.x installed from the MouseLand GitHub HEAD. See [Set up the CPSAM environment](https://docs.elembio.io/docs/tutorials/cytoprofiling/custom-segmentation/#cpsam-setup-env). Do not install Cellpose 4.x into your `cytoprofiling-seg` environment.
- You set up the `cytoprofiling-seg` Python environment with **Cellpose 3.x**. This is **required** for Step 7 (nuclear segmentation). The Element Biosciences nuclear model is a Cellpose 3 model and cannot load in the Cellpose 4 CPSAM environment, so nuclear segmentation runs in a subprocess using the `cytoprofiling-seg` interpreter. If you have not set up this environment yet, create it now:
  ```bash
  python -m venv venv-cytoprofiling-seg
  source venv-cytoprofiling-seg/bin/activate  # On Windows: venv-cytoprofiling-seg\Scripts\activate
  pip install cellpose==3.0.7 numpy==1.26.4 pandas==2.2.3 scikit-image==0.24.0 Pillow==10.4.0
  ```
  Then update the `seg_python` variable in **Step 2** to the full path of this environment's Python interpreter (e.g. `/path/to/venv-cytoprofiling-seg/bin/python`).
- The `cpsam` environment is selected as this notebook's kernel (top menu: **Kernel → Change kernel → CellposeSAM**).
- A GPU is available. CPSAM on CPU is prohibitively slow for full-run processing.
- You know your run type:
  - **Teton / Teton Atlas** runs have an actin channel — use **3-channel** mode.
  - **Cell Paint only** runs lack an actin channel — use **2-channel** mode.

> **First run downloads ~1.15 GB.** The first time you initialize the CPSAM model, Cellpose automatically downloads the model weights (~1.15 GB) from HuggingFace to `~/.cellpose/models/`. Subsequent runs use the cached weights and do not require an internet connection.

## Step 1 — Import packages

Load the imaging, numerics, and Cellpose packages used throughout the rest of the notebook.

In [ ]:
import json
import os

import numpy as np
import skimage
from cellpose import core, models, transforms


def normalize_image(image, region_size=1824):
    image_norm = np.zeros_like(image, np.single)
    for xi in range(int(image.shape[1] / region_size)):
        for yi in range(int(image.shape[0] / region_size)):
            cropped = image[
                yi * region_size: (yi + 1) * region_size,
                xi * region_size: (xi + 1) * region_size,
            ]
            cropped = transforms.normalize_img(
                cropped.reshape(cropped.shape[0], cropped.shape[1], 1)
            ).reshape(cropped.shape[0], cropped.shape[1])
            image_norm[
                yi * region_size: (yi + 1) * region_size,
                xi * region_size: (xi + 1) * region_size,
            ] = cropped
    return image_norm


## Step 2 — Provide Input and Output Paths

Set the required paths below. Use a fresh `output_location` per re-segmentation pass so CPSAM masks do not overwrite Element Biosciences masks.

- `model_dir` and `nuclear_model` point to the Element Biosciences nuclear segmentation model.
- `seg_python` is the path to the `cytoprofiling-seg` venv Python interpreter (Cellpose 3). The nuclear model is a Cellpose 3 model and cannot load in the Cellpose 4 CPSAM environment, so nuclear segmentation runs in a subprocess.

In [ ]:
# Edit all paths before running the rest of the notebook.

# Path to your AVITI24 run output folder
run_directory   = r"/path/to/your/Run/Output/Folder"

# Where to write the CPSAM segmentation mask outputs (must be a different folder)
output_location = r"/path/to/your/Run/Output/Folder/Segmentation_Output"

# Path to segmentation models (for the nuclear model)
model_dir       = r"/path/to/cytoprofiling/src/segmentationModels"
nuclear_model   = "20250212_cellpose_nuc_8diam"

# Path to the cytoprofiling-seg venv Python (Cellpose 3, required for the nuclear model).
# The nuclear model is a Cellpose 3 model and cannot run in the Cellpose 4 (CPSAM)
# environment, so nuclear segmentation runs in a subprocess using this interpreter.
seg_python      = r"/path/to/git/cytoprofiling/src/python/examples/segmentation_workbook/venv-cytoprofiling-seg/bin/python"

## Step 3 — Choose channel mode and normalization

Set `num_channels` to match your run type:
- **3** for Teton / Teton Atlas runs (Cell-Membrane + Nucleus + Actin).
- **2** for Cell Paint only runs (Cell-Membrane + Nucleus only; no actin channel).

Set `normalize` to `True` (default) to apply region-based intensity normalization before CPSAM inference. Set to `False` to pass raw images directly to the model.

In [ ]:
# Set to 3 for Teton/Teton Atlas runs, or 2 for Cell Paint only runs.
num_channels = 3

## Step 4 — Confirm GPU and load CPSAM

Verify that a GPU is available, then load the CPSAM model. The first time this cell runs, Cellpose downloads ~1.15 GB of model weights from HuggingFace to `~/.cellpose/models/`. Subsequent runs use the cached weights.

In [ ]:
if not core.use_gpu():
    print("WARNING: No GPU detected. Runtime will be very long (8+ hours for 12-well).")
    print("Consider running on a GPU-equipped machine for production use.")
else:
    print("GPU confirmed. Proceeding with CPSAM segmentation.")

# Load model. Downloads on first run (~1.15 GB).
model = models.CellposeModel(gpu=True)


## Step 5 — Build the tile list from `RunParameters.json`

Read `RunParameters.json` to enumerate every well and tile in the run and build the `tile2well` map used by the segmentation loop. The cell prints the total tile count so you can confirm the workload before committing to Steps 6–7.

In [ ]:
with open(os.path.join(run_directory, "RunParameters.json")) as f:
    run_parameters = json.load(f)

tile2well, tiles = {}, []
for well in run_parameters["Wells"]:
    for tile in well["Tiles"]:
        tile2well[tile["Name"]] = well["WellLocation"]
        tiles.append(tile["Name"])


print(f"Total tiles to process: {len(tiles)}")


## Step 6 — Cell segmentation (CPSAM)

Run CPSAM on each tile and write the cell masks to `output_location/Well{well}/`. Unlike the Element Biosciences workflow, CPSAM uses a single model for every well, so no per-well model lookup is needed. The loop builds a 3-channel or 2-channel composite based on the `num_channels` setting from Step 3.

To monitor progress, watch for the rolling `Done: ...` lines. Each line corresponds to one tile fully processed and saved. See the **Runtime expectations** table at the bottom of the notebook for typical wall-clock times.

In [ ]:
flow_threshold      = 0.4
cellprob_threshold  = 0.0

print(f"Beginning {num_channels}-channel CPSAM cell segmentation across {len(tiles)} tiles")

for tile in tiles:
    well = tile2well[tile]
    os.makedirs(os.path.join(output_location, f"Well{well}"), exist_ok=True)

    cell_image    = skimage.io.imread(
        os.path.join(run_directory, "Projection", f"Well{well}", f"CP01_{tile}_Cell-Membrane.tif")
    )
    nuclear_image = skimage.io.imread(
        os.path.join(run_directory, "Projection", f"Well{well}", f"CP01_{tile}_Nucleus.tif")
    )

    cell_image    = normalize_image(cell_image)
    nuclear_image = normalize_image(nuclear_image)

    if num_channels == 3:
        actin_image = skimage.io.imread(
            os.path.join(run_directory, "Projection", f"Well{well}", f"CP01_{tile}_Actin.tif")
        )
        actin_image = normalize_image(actin_image)
        composite = np.stack([cell_image, nuclear_image, actin_image], axis=-1)
    else:
        composite = np.stack([cell_image, nuclear_image], axis=-1)

    print(f"Segmenting cell membrane \u2014 tile: {tile}")
    cell_mask, _, _ = model.eval(
        composite,
        batch_size=2,
        normalize=False,
        flow_threshold=flow_threshold,
        cellprob_threshold=cellprob_threshold,
        resample=False,
    )
    cell_mask = cell_mask.astype(np.uint32)

    skimage.io.imsave(
        os.path.join(output_location, f"Well{well}", f"{tile}_Cell.tif"),
        cell_mask.astype(np.uint16),
    )
    print(f"Done: {tile}")

print(f"\nCell segmentation complete. Proceed to Step 7 for nuclear segmentation.")


## Step 7 — Nuclear segmentation (Element Biosciences model via subprocess)

The Element Biosciences nuclear model is a Cellpose 3 model that cannot load in the Cellpose 4 (CPSAM) environment. This step runs nuclear segmentation in a subprocess using the `cytoprofiling-seg` venv (Cellpose 3). Tiles are processed in batches to balance memory and throughput.

The subprocess loads the nuclear model once per batch, runs inference on every tile in the batch, and writes the binary nuclear masks to `output_location`.

In [ ]:
import subprocess, textwrap, shutil

if not shutil.which(seg_python) and not os.path.isfile(seg_python):
    raise FileNotFoundError(
        f"cytoprofiling-seg Python not found at: {seg_python}\n"
        "Update 'seg_python' in Step 2 to point to your cytoprofiling-seg venv interpreter."
    )

nuclear_model_path = os.path.join(model_dir, nuclear_model)
if not os.path.exists(nuclear_model_path):
    raise FileNotFoundError(f"Nuclear model not found at: {nuclear_model_path}")

BATCH_SIZE = 24

_NUC_SCRIPT = textwrap.dedent("""\
    import sys, json, os
    import numpy as np
    import skimage.io
    from cellpose import models, transforms

    def normalize_image(image, region_size=1824):
        image_norm = np.zeros_like(image, np.single)
        for xi in range(int(image.shape[1] / region_size)):
            for yi in range(int(image.shape[0] / region_size)):
                cropped = image[
                    yi * region_size: (yi + 1) * region_size,
                    xi * region_size: (xi + 1) * region_size,
                ]
                cropped = transforms.normalize_img(
                    cropped.reshape(cropped.shape[0], cropped.shape[1], 1)
                ).reshape(cropped.shape[0], cropped.shape[1])
                image_norm[
                    yi * region_size: (yi + 1) * region_size,
                    xi * region_size: (xi + 1) * region_size,
                ] = cropped
        return image_norm

    config = json.loads(sys.argv[1])
    model_path    = config["model_path"]
    use_gpu       = config["use_gpu"]
    tiles         = config["tiles"]
    run_dir       = config["run_dir"]
    out_dir       = config["out_dir"]
    tile2well     = config["tile2well"]

    nuc_model = models.CellposeModel(
        gpu=use_gpu, pretrained_model=False, model_type=model_path
    )

    for tile in tiles:
        well = tile2well[tile]
        nuc_img = skimage.io.imread(
            os.path.join(run_dir, "Projection", f"Well{well}", f"CP01_{tile}_Nucleus.tif")
        )
        nuc_img = normalize_image(nuc_img)
        mask, _, _ = nuc_model.eval(nuc_img, resample=False)
        binary = (mask > 0).astype(np.uint8)

        cell_mask = skimage.io.imread(
            os.path.join(out_dir, f"Well{well}", f"{tile}_Cell.tif")
        )
        binary[cell_mask == 0] = 0

        skimage.io.imsave(
            os.path.join(out_dir, f"Well{well}", f"{tile}_Nuclear.tif"), binary
        )
        print(f"Nuclear done: {tile}", flush=True)
""")

use_gpu_for_nuc = core.use_gpu()
total = len(tiles)

print(f"Beginning nuclear segmentation across {total} tiles (batch size {BATCH_SIZE})")
print(f"Model: {nuclear_model}  |  GPU: {use_gpu_for_nuc}")

for batch_start in range(0, total, BATCH_SIZE):
    batch = tiles[batch_start : batch_start + BATCH_SIZE]
    batch_num = batch_start // BATCH_SIZE + 1
    num_batches = (total + BATCH_SIZE - 1) // BATCH_SIZE
    print(f"\n--- Batch {batch_num}/{num_batches} ({len(batch)} tiles) ---")

    config = json.dumps({
        "model_path": nuclear_model_path,
        "use_gpu":    use_gpu_for_nuc,
        "tiles":      batch,
        "run_dir":    run_directory,
        "out_dir":    output_location,
        "tile2well":  {t: tile2well[t] for t in batch},
    })

    result = subprocess.run(
        [seg_python, "-c", _NUC_SCRIPT, config],
        capture_output=True, text=True,
    )

    if result.stdout:
        print(result.stdout, end="")
    if result.returncode != 0:
        print(f"ERROR in batch {batch_num}:\n{result.stderr}")
        raise RuntimeError(f"Nuclear segmentation subprocess failed (batch {batch_num})")

print(f"\nNuclear segmentation complete for all {total} tiles.")

## Reference

### Runtime expectations

Cell segmentation (Step 6, CPSAM) and nuclear segmentation (Step 7, Element Biosciences model) run sequentially. CPU runtimes for CPSAM are prohibitive; the table below assumes a GPU. Nuclear segmentation is much faster.

| Plate format | Approximate tiles | CPSAM cell (GPU) | Nuclear (GPU) |
| ------------ | ----------------- | ---------------- | ------------- |
| 1-well       | ~18 tiles         | ~30 minutes      | ~5 minutes    |
| 12-well      | ~216 tiles        | ~6–10 hours      | ~30 minutes   |
| 48-well      | ~864 tiles        | ~24–36 hours     | ~2 hours      |

### Output files

For each tile, Steps 6 and 7 write two files to your `output_location`:

- `{tile}_Cell.tif` (Step 6): a `uint16` label mask where each unique integer represents one segmented cell.
- `{tile}_Nuclear.tif` (Step 7): a `uint8` binary mask where `0` indicates no nucleus and `1` indicates a nucleus is present.

### Why a subprocess for nuclear segmentation?

The Element Biosciences nuclear model is trained with Cellpose 3. Cellpose 4 (used by CPSAM) cannot load Cellpose 3 models — they are architecturally incompatible. To use both models in one notebook, Step 7 spawns a subprocess that runs the `cytoprofiling-seg` Python interpreter (Cellpose 3) to perform nuclear inference. The subprocess loads the model once per batch and processes all tiles in that batch before exiting.

### Validation

CPSAM does not produce a built-in quality metrics table. Verify outputs visually or by comparing cell and nucleus counts against the baseline you established in [Interpret results and choose a model](https://docs.elembio.io/docs/tutorials/cytoprofiling/custom-segmentation/#interpret-results).

### Third-party tool disclaimer

CellposeSAM is provided by the [MouseLand open-source project](https://github.com/mouseland/cellpose) and is not affiliated with or endorsed by Element Biosciences. CPSAM has not been formally validated against AVITI24 cytoprofiling runs and results may vary. For CPSAM-specific issues, installation support, or model updates, refer to the [official MouseLand repository](https://github.com/mouseland/cellpose).

After all tiles finish, re-run Cells2Stats with `--segmentation` pointing at `output_location` to regenerate the cell table. See [Re-run Cells2Stats for cell assignment](https://docs.elembio.io/docs/tutorials/cytoprofiling/custom-segmentation/#cell-assignment).